In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "src").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("找不到项目根目录（包含 src 的目录）")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)

PROJECT_ROOT = /home/joyce/projects/cross-asset-quant-lab/cryptoAlpha


In [2]:
import pandas as pd
import numpy as np

from src.factors.factor_builder import FactorBuilder
from src.models.ridge_alpha_model import RidgeAlphaModel
from src.portfolio.portfolio_backtest import (
    backtest_long_short_portfolio,
    summarize_portfolio_result,
    build_execution_target_table,
)
from src.portfolio.portfolio_plotter import PortfolioPlotter

In [3]:
factor_names = [
    "mom_24h",
    "mom_6h",
    "funding_z_24",
    "oi_change_24h",
    "taker_imbalance",
    "long_short_ratio_z_24",
    "volume_ratio_24",
    "active_community_count_z_24",
]
import pandas as pd
cache_dir = PROJECT_ROOT / "data" / "cache"
cache_dir.mkdir(parents=True, exist_ok=True)

panel_fp = cache_dir / "panel_top20_1h_2025_20260311.parquet"
panel_df = pd.read_parquet(panel_fp)
builder = FactorBuilder()
factor_df = builder.compute_many(
    panel_df,
    factor_names
)

print(factor_df.head())
print(factor_df.shape)

/home/joyce/projects/cross-asset-quant-lab/cryptoAlpha/src/factors/factor_utils.py:32: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  return df.groupby("symbol")[col].pct_change(periods)


    datetime    symbol  mom_24h  mom_6h  funding_z_24  oi_change_24h  \
0 2025-01-01   ADAUSDT      NaN     NaN           NaN            NaN   
1 2025-01-01   APTUSDT      NaN     NaN           NaN            NaN   
2 2025-01-01  ATOMUSDT      NaN     NaN           NaN            NaN   
3 2025-01-01  AVAXUSDT      NaN     NaN           NaN            NaN   
4 2025-01-01   BCHUSDT      NaN     NaN           NaN            NaN   

   taker_imbalance  long_short_ratio_z_24  volume_ratio_24  \
0         0.224374                    NaN              NaN   
1         0.117690                    NaN              NaN   
2         0.014625                    NaN              NaN   
3         0.151342                    NaN              NaN   
4         0.078970                    NaN              NaN   

   active_community_count_z_24  
0                          NaN  
1                          NaN  
2                          NaN  
3                          NaN  
4                          Na

In [ ]:
model = RidgeAlphaModel(
    horizon=1,       # 预测未来1h收益
    train_window=180, # 训练窗口，按时间点滚动
    alpha=1.0,
)

# 先在 panel 上构造 label
df_model = model.build_label(panel_df)

# 再 merge 因子
df_model = df_model.merge(
    factor_df,
    on=["datetime", "symbol"],
    how="left",
)

print(df_model.shape)
df_model.head()

(197923, 36)


,datetime,symbol,open,high,low,close,volume_usd,funding_open,funding_high,funding_low,...,negative_sentiment_ratio,future_return,mom_24h,mom_6h,funding_z_24,oi_change_24h,taker_imbalance,long_short_ratio_z_24,volume_ratio_24,active_community_count_z_24
0,2025-01-01,ADAUSDT,0.8448,0.8607,0.8434,0.8593,1.443051e+07,0.01,0.01,0.01,...,NaN,0.100314,NaN,NaN,NaN,NaN,0.224374,NaN,NaN,NaN
1,2025-01-01,APTUSDT,8.7120,8.8049,8.7014,8.8004,5.540888e+06,0.01,0.01,0.01,...,NaN,0.036998,NaN,NaN,NaN,NaN,0.117690,NaN,NaN,NaN
2,2025-01-01,ATOMUSDT,6.1870,6.2810,6.1770,6.2790,1.511668e+06,0.01,0.01,0.01,...,NaN,0.063227,NaN,NaN,NaN,NaN,0.014625,NaN,NaN,NaN
3,2025-01-01,AVAXUSDT,35.6890,36.2210,35.6310,36.1900,8.185929e+06,0.01,0.01,0.01,...,NaN,0.063830,NaN,NaN,NaN,NaN,0.151342,NaN,NaN,NaN
4,2025-01-01,BCHUSDT,434.3500,440.7100,433.7300,440.5900,3.551392e+06,0.01,0.01,0.01,...,NaN,0.036746,NaN,NaN,NaN,NaN,0.078970,NaN,NaN,NaN


In [9]:
pred_df = model.fit_predict(
    df=df_model,
    feature_cols=factor_names,
    label_col="future_return",
)

print(pred_df.shape)
pred_df.head()

KeyboardInterrupt: 

In [ ]:
pred_dir = PROJECT_ROOT / "data" / "predictions"
pred_dir.mkdir(parents=True, exist_ok=True)

pred_fp = pred_dir / "pred_df_top20_1h_2025_20260311.parquet"
pred_df.to_parquet(pred_fp, index=False)

print("saved pred_df to:", pred_fp)
pred_df.head()

In [ ]:
bt_result = backtest_long_short_portfolio(
    pred_df=pred_df,
    panel_df=panel_df,
    quantile=0.1,
    rebalance_every_hours=24,
    portfolio_forward_hours=24,
)

summary = pd.Series(summarize_portfolio_result(bt_result), name="value")
summary

In [ ]:
execution_dir = PROJECT_ROOT / "data" / "execution"
execution_dir.mkdir(parents=True, exist_ok=True)

btc_execution_df = build_execution_target_table(
    bt_result=bt_result,
    panel_df=panel_df,
    symbol="BTCUSDT",
    initial_portfolio_value=1_000_000.0,
)

btc_execution_1226_1228 = (
    btc_execution_df[
        btc_execution_df["datetime"].between("2025-12-26", "2025-12-28 23:59:59")
    ]
    .rename(
        columns={
            "current_weight": "BTC current_weight",
            "target_weight": "BTC target_weight",
        }
    )
    .reset_index(drop=True)
)

btc_execution_fp = execution_dir / "btc_execution_targets_20251226_20251228.csv"
btc_execution_1226_1228.to_csv(btc_execution_fp, index=False)

print("saved btc execution targets to:", btc_execution_fp)
btc_execution_1226_1228.head(20)

In [ ]:
execution_root = PROJECT_ROOT.parent / "execution_rl_project"
chunk_root = Path("/home/joyce/projects/data/raw/tardis_chunks") / "BTCUSDT"
target_days = {"2025-12-26", "2025-12-27", "2025-12-28"}

chunk_records = []
for chunk_index, chunk_dir in enumerate(sorted(p for p in chunk_root.iterdir() if p.is_dir())):
    chunk_name = chunk_dir.name
    if chunk_name[:10] in target_days:
        chunk_records.append(
            {
                "chunk_index": chunk_index,
                "chunk": chunk_name,
                "book_path": str(chunk_dir / "book.parquet"),
                "trade_path": str(chunk_dir / "trades.parquet"),
                "snapshot_path": str(chunk_dir / "snapshot.parquet"),
            }
        )

chunk_list_df = pd.DataFrame(chunk_records)
selected_chunk_df = (
    chunk_list_df[
        chunk_list_df["chunk"].isin(["2025-12-26_18", "2025-12-27_18", "2025-12-28_00"])
    ]
    .reset_index(drop=True)
)

chunk_list_fp = execution_dir / "btc_chunk_list_20251226_20251228.csv"
chunk_list_df.to_csv(chunk_list_fp, index=False)

print("saved chunk list to:", chunk_list_fp)
selected_chunk_df

In [ ]:
artifact_records = [
    {
        "artifact": "train_config",
        "path": str(execution_root / "configs" / "train_btc_long.yaml"),
    },
    {
        "artifact": "buy_checkpoint",
        "path": str(execution_root / "results" / "checkpoints_btcusdt_buy_1m" / "btcusdt_buy_1m.zip"),
    },
    {
        "artifact": "buy_vecnormalize",
        "path": str(execution_root / "results" / "checkpoints_btcusdt_buy_1m" / "btcusdt_buy_1m_vecnormalize.pkl"),
    },
    {
        "artifact": "sell_checkpoint",
        "path": str(execution_root / "results" / "checkpoints_btcusdt_sell_1m" / "btcusdt_sell_1m.zip"),
    },
    {
        "artifact": "sell_vecnormalize",
        "path": str(execution_root / "results" / "checkpoints_btcusdt_sell_1m" / "btcusdt_sell_1m_vecnormalize.pkl"),
    },
    {
        "artifact": "buy_train_log",
        "path": str(execution_root / ".copilot_run_logs" / "btcusdt_buy" / "step3_train_both.log"),
    },
    {
        "artifact": "sell_train_log",
        "path": str(execution_root / ".copilot_run_logs" / "btcusdt_sell" / "step3_train_both.log"),
    },
]

artifact_df = pd.DataFrame(artifact_records)
artifact_df["exists"] = artifact_df["path"].map(lambda x: Path(x).exists())
artifact_df

In [ ]:
import re

def parse_execution_eval_log(log_path: Path) -> pd.DataFrame:
    lines = log_path.read_text().splitlines()
    meta = {"log_path": str(log_path)}
    rows = []
    current_row = None

    for line in lines:
        text = line.strip()
        if not text:
            continue

        if text.startswith("side:"):
            meta["side"] = text.split(":", 1)[1].strip()
            continue

        if text.startswith("chunk:"):
            meta["chunk"] = text.split(":", 1)[1].strip()
            continue

        if text.endswith(":") and not text.startswith("==="):
            current_row = {
                "strategy": text[:-1],
                "side": meta.get("side"),
                "chunk": meta.get("chunk"),
                "log_path": meta["log_path"],
            }
            continue

        if current_row is None:
            continue

        if text.startswith("episodes="):
            current_row["episodes"] = int(text.split("=", 1)[1])
            continue

        match = re.match(r"avg_reward=([-0-9.]+) \| std_reward=([-0-9.]+)", text)
        if match:
            current_row["avg_reward"] = float(match.group(1))
            current_row["std_reward"] = float(match.group(2))
            continue

        match = re.match(r"avg_filled=([-0-9.]+) \| std_filled=([-0-9.]+)", text)
        if match:
            current_row["avg_filled"] = float(match.group(1))
            current_row["std_filled"] = float(match.group(2))
            continue

        match = re.match(r"avg_remaining=([-0-9.]+) \| std_remaining=([-0-9.]+)", text)
        if match:
            current_row["avg_remaining"] = float(match.group(1))
            current_row["std_remaining"] = float(match.group(2))
            continue

        match = re.match(r"avg_equity=([-0-9.]+) \| std_equity=([-0-9.]+)", text)
        if match:
            current_row["avg_equity"] = float(match.group(1))
            current_row["std_equity"] = float(match.group(2))
            rows.append(current_row.copy())

    return pd.DataFrame(rows)

eval_log_paths = [
    execution_root / ".copilot_run_logs" / "btcusdt_buy" / "eval_buy_2025-12-26_18_50episodes.log",
    execution_root / ".copilot_run_logs" / "btcusdt_buy" / "eval_buy_2025-12-27_18_20episodes.log",
    execution_root / ".copilot_run_logs" / "btcusdt_buy" / "eval_buy_2025-12-28_00_20episodes.log",
    execution_root / ".copilot_run_logs" / "btcusdt_sell" / "eval_sell_2025-12-26_18_20episodes.log",
    execution_root / ".copilot_run_logs" / "btcusdt_sell" / "eval_sell_2025-12-27_18_20episodes.log",
    execution_root / ".copilot_run_logs" / "btcusdt_sell" / "eval_sell_2025-12-28_00_20episodes.log",
]

execution_eval_df = pd.concat(
    [parse_execution_eval_log(path) for path in eval_log_paths if path.exists()],
    ignore_index=True,
)

execution_eval_fp = execution_dir / "execution_evaluation_20251226_20251228.csv"
execution_eval_df.to_csv(execution_eval_fp, index=False)

print("saved execution evaluation to:", execution_eval_fp)
execution_eval_df